In [1]:
import os
os.getcwd()

'C:\\Users\\PC'

In [2]:
os.chdir(r"D:\FQL\PJ 4")

### Section 1 : Import Libraries

In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings, os, joblib
warnings.filterwarnings('ignore')

In [4]:
from statsmodels.tsa.statespace.sarimax import SARIMAX
from prophet import Prophet
from sklearn.ensemble import RandomForestRegressor

In [5]:
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import TimeSeriesSplit, RandomizedSearchCV
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error
import xgboost as xgb

In [6]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam

In [7]:
os.makedirs('models',      exist_ok=True)
os.makedirs('predictions', exist_ok=True)
os.makedirs('plots',       exist_ok=True)

In [8]:
C = dict(blue='#1D4ED8', green='#15803D', red='#B91C1C', amber='#B45309', violet='#6D28D9')
plt.rcParams.update({'figure.dpi': 120, 'axes.spines.top': False, 'axes.spines.right': False})
print('Ready.')

Ready.


### Section 2 :Load Preprocessed Data

In [ ]:
df = pd.read_csv('data/processed_traffic.csv', parse_dates=['Date'])
df = df.sort_values('Date').reset_index(drop=True)

In [ ]:
NUM_COLS = ['Page_Loads', 'Unique_Visits', 'First_Time_Visits', 'Returning_Visits']
TARGET   = 'Page_Loads'
print(f'Loaded: {df.shape}  |  {df["Date"].min().date()} to {df["Date"].max().date()}')

In [ ]:
df.head()

### Section 3 : Feature Engineering

In [ ]:
df_feat = df.copy()

In [ ]:
# Calendar Features
df_feat['day_of_week']  = df_feat['Date'].dt.dayofweek
df_feat['day_of_month'] = df_feat['Date'].dt.day
df_feat['month']        = df_feat['Date'].dt.month
df_feat['week_of_year'] = df_feat['Date'].dt.isocalendar().week.astype(int)
df_feat['quarter']      = df_feat['Date'].dt.quarter
df_feat['year']         = df_feat['Date'].dt.year
df_feat['is_weekend']   = (df_feat['day_of_week'] >= 5).astype(int)
df_feat['is_month_end'] = df_feat['Date'].dt.is_month_end.astype(int)
df_feat['is_month_start'] = df_feat['Date'].dt.is_month_start.astype(int)

In [ ]:
# Fourier terms (capture weekly & yearly cycles)
for k in [1, 2]:
    df_feat[f'sin_week_{k}'] = np.sin(2 * np.pi * k * df_feat['day_of_week'] / 7)
    df_feat[f'cos_week_{k}'] = np.cos(2 * np.pi * k * df_feat['day_of_week'] / 7)

In [ ]:
# Lag Features (Page_Loads)
for lag in [1, 2, 3, 7, 14, 21, 28]:
    df_feat[f'lag_{lag}'] = df_feat[TARGET].shift(lag)

In [ ]:
# Rolling Statistics
for window in [7, 14, 30]:
    shifted = df_feat[TARGET].shift(1)
    df_feat[f'roll_mean_{window}'] = shifted.rolling(window).mean()
    df_feat[f'roll_std_{window}']  = shifted.rolling(window).std()
    df_feat[f'roll_max_{window}']  = shifted.rolling(window).max()
    df_feat[f'roll_min_{window}']  = shifted.rolling(window).min()

In [ ]:
# Expanding mean (global trend signal)
df_feat['exp_mean'] = df_feat[TARGET].shift(1).expanding().mean()

# Visitor Ratio Features
df_feat['return_ratio'] = df_feat['Returning_Visits']   / (df_feat['Unique_Visits'] + 1)
df_feat['new_ratio']    = df_feat['First_Time_Visits']  / (df_feat['Unique_Visits'] + 1)
df_feat['loads_per_visit'] = df_feat[TARGET] / (df_feat['Unique_Visits'] + 1)

In [ ]:
# Lag features for other traffic metrics
for col in ['Unique_Visits', 'Returning_Visits']:
    df_feat[f'{col}_lag1'] = df_feat[col].shift(1)
    df_feat[f'{col}_lag7'] = df_feat[col].shift(7)

In [ ]:
# Drop NaN rows created by lagging
df_feat = df_feat.dropna().reset_index(drop=True)
print(f'Feature-engineered shape: {df_feat.shape}')

In [ ]:
FEAT_COLS = [c for c in df_feat.columns
             if c not in ['Date', 'DayName', TARGET] + NUM_COLS]
print(f'Total features: {len(FEAT_COLS)}')
print(FEAT_COLS)